# Efficiency diagnostics for Υ at 9×275 GeV

This companion notebook runs the same separate, read-only diagnostic as the command-line script. It does not modify notebooks 01–03 or previous framework outputs. It checks the Q² split, matched-Q² electron-ID comparison, raw tail counts, selection losses and pT correlations. Numerical results are finite-MC efficiencies, with no luminosity projection.

Keep this notebook beside `run_efficiency_diagnostics.py` and `diagnostics/`. Use a fresh run ID. The default ROOT paths are in `config_9x275.json`; edit a copy and change CONFIG if needed. The results must be reviewed before claiming a detector mechanism.


New runs default to **proton tagged**, requiring `reco_topology` 1 or 2 and positive finite `reco_tagged_proton_p`. The generated truth denominator is unchanged. Set `PROTON_SELECTION="inclusive"` only to reproduce the previous category. Before/after counts are raw MC yields, with no luminosity scaling. Use the new run ID so previous outputs remain preserved.


This separate proton-tagged copy writes only to `Ypsilon-SWAN-ProtonTagged` by default. The earlier no-tag notebooks and results are preserved. The proton requirement is implemented and tested in this code; old figures are not regenerated until you execute a fresh run. These efficiencies use unweighted MC counts and finite-MC Wilson intervals; no luminosity, cross section, photon flux or weight branch enters.


In [ ]:
from pathlib import Path
from IPython.display import display, Image, Markdown, FileLink
import pandas as pd
from run_efficiency_diagnostics import main

FRAMEWORK_ROOT = Path("/eos/user/g/gkrintir/Ypsilon-SWAN-ProtonTagged")
CONFIG = Path.cwd() / "config_9x275.json"
RUN_ID = "efficiency-9x275-proton-v3"
PROTON_SELECTION = "required"  # inclusive reproduces the old category
BASE_ARGS = ["--framework-root", str(FRAMEWORK_ROOT), "--config", str(CONFIG), "--proton-selection", PROTON_SELECTION]


## Check the inputs

The supplied config uses 1S `output_newID` and `output`, plus 2S/3S `output`. Validation checks the input schema and every topology/momentum pair for the required proton category. It requires access to the relevant EOS files from this session.


In [ ]:
main(["plan"] + BASE_ARGS)
status = main(["validate"] + BASE_ARGS)
if status != 0:
    raise RuntimeError("Input validation failed. Check the paths above before running.")


## Run into a new directory

Existing run IDs are refused. Successful samples are retained if another sample fails; PARTIAL status must be resolved or explicitly retained in interpretation. COMPLETE means the diagnostic executed, not that the physical cause has been validated.


In [ ]:
status = main(["run"] + BASE_ARGS + ["--run-id", RUN_ID, "--hash-inputs"])
print("Exit status:", status)
OUTPUT = FRAMEWORK_ROOT / "results/generated/runs" / RUN_ID / "artifacts/efficiency_diagnostics"
print("Output:", OUTPUT)


## Review the evidence

Read the execution/reference-check status first. The Q² mixture plot tests whether a changing population contributes to the inclusive trend. The matched-ID plots require production metadata checks; displayed Wilson intervals are individual intervals, with no assumption that the scenarios are independent.


In [ ]:
display(Markdown((OUTPUT / "summary.md").read_text()))
for name in ["diagnostic_status.json", "sample_audits.json", "input_comparability.json"]:
    display(FileLink(str(OUTPUT / name)))
for stem in ["1s_newid_q2_split", "1s_newid_q2_mixture", "1s_matched_q2_id_comparison", "1s_high_pt_counts"]:
    image = OUTPUT / "plots" / (stem + ".png")
    if image.exists():
        display(Image(filename=str(image)))


In [ ]:
eff = pd.read_csv(OUTPUT / "tables/efficiency.csv")
display(eff.loc[(eff["sample"] == "1s_newid") & (eff["variable"] == "true_pT") & (eff["low"] >= 1.6 - 1e-10)])
mixture = OUTPUT / "tables/q2_mixture.csv"
if mixture.exists():
    display(pd.read_csv(mixture).query("sample == '1s_newid' and variable == 'true_pT'"))
